# Streaming CKA: comparing representations on streamed data

TorchLens ships streaming accumulators in `tl.stats` — `CKA` (linear centered
kernel alignment, Kornblith et al. 2019) and `CrossCovariance` — that retain only
**feature-sized** state. You can compare representations across an arbitrarily
long data stream without ever materializing the full `(n_observations, n_features)`
activation stack: feed each batch's activations with `.update(a, b)` and read the
finalized value with `.result()` at the end.

This notebook demonstrates the pattern end to end:

1. stream batches through a model, capturing two layers per batch with `tl.trace`;
2. update a `CKA` and a `CrossCovariance` accumulator per batch;
3. verify the streamed CKA matches the one-shot `tl.stats.cka` on the
   concatenated activations;
4. build a layer-by-layer CKA table between two models on the same stream.

In [ ]:
import torch
from torch import nn

import torchlens as tl

torch.manual_seed(0)


class SmallMlp(nn.Module):
    """Two-hidden-layer MLP used for the representation comparison."""

    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(16, 32)
        self.fc2 = nn.Linear(32, 32)
        self.head = nn.Linear(32, 4)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h1 = torch.relu(self.fc1(x))
        h2 = torch.relu(self.fc2(h1))
        return self.head(h2)


model = SmallMlp().eval()


def batch_stream(n_batches: int = 30, batch_size: int = 64, seed: int = 1):
    """Yield deterministic input batches, simulating a data loader."""
    generator = torch.Generator().manual_seed(seed)
    for _ in range(n_batches):
        yield torch.randn(batch_size, 16, generator=generator)


print(model)

## Stream once, accumulate as you go

Per batch we capture just the two layers we care about (`save=` keeps the trace
sparse), update the accumulators, and drop the trace. Only the accumulators'
feature-sized state survives the loop. For this demo we also keep the raw
activations in lists so the next cell can verify the streamed result against
the one-shot computation — in real use at scale you would keep only the
accumulators.

In [ ]:
streaming_cka = tl.stats.CKA(name="relu1-vs-relu2")
cross_covariance = tl.stats.CrossCovariance(name="relu1-x-relu2")

kept_a, kept_b = [], []  # verification only; not needed for streaming use
for batch in batch_stream():
    log = tl.trace(model, batch, save=tl.func("relu"))
    act_a = log["relu_1_2"].out
    act_b = log["relu_2_4"].out
    streaming_cka.update(act_a, act_b)
    cross_covariance.update(act_a, act_b)
    kept_a.append(act_a)
    kept_b.append(act_b)

print(f"streamed linear CKA(relu1, relu2) = {streaming_cka.result():.6f}")
print("cross-covariance shape:", tuple(cross_covariance.result().shape))

## The streamed value is exact, not an approximation

`CKA.update` maintains exact running (cross-)covariances, so the streamed result
equals the one-shot `tl.stats.cka` on the concatenated activations up to
floating-point accumulation order.

In [ ]:
full_a = torch.cat(kept_a)
full_b = torch.cat(kept_b)
one_shot = tl.stats.cka(full_a, full_b)
streamed = streaming_cka.result()
print(f"one-shot: {one_shot:.12f}")
print(f"streamed: {streamed:.12f}")
assert abs(one_shot - streamed) < 1e-9, "streamed CKA diverged from one-shot"
print("MATCH")

## Layer-by-layer CKA between two models on one stream

The same pattern scales to a grid of accumulators: here every (layer, layer)
pair between a trained-init model and an independent random init, updated from
one shared stream. Each accumulator still holds only feature-sized state.

In [ ]:
torch.manual_seed(42)
other_model = SmallMlp().eval()

layers = ["relu_1_2", "relu_2_4", "linear_3_5"]  # linear_3_5 = the head projection


def activations(log: tl.Trace) -> dict[str, torch.Tensor]:
    """Return the three compared activations from one sparse trace."""
    return {name: log[name].out for name in layers}


grid = {(a, b): tl.stats.CKA() for a in layers for b in layers}
for batch in batch_stream(n_batches=20, seed=2):
    log_self = tl.trace(model, batch, save=tl.func("relu") | tl.func("linear"))
    log_other = tl.trace(other_model, batch, save=tl.func("relu") | tl.func("linear"))
    acts_self = activations(log_self)
    acts_other = activations(log_other)
    for a in layers:
        for b in layers:
            grid[(a, b)].update(acts_self[a], acts_other[b])

header = " " * 12 + "".join(f"{b:>12}" for b in layers)
print("rows: model A layers | columns: model B layers")
print(header)
for a in layers:
    row = "".join(f"{grid[(a, b)].result():12.4f}" for b in layers)
    print(f"{a:>12}{row}")

## Notes

- `CKA.update(a, b)` requires **paired batches**: the same rows (observations)
  must be fed to both sides in the same order, and feature dimensions must stay
  constant across updates. A zero-variance side makes `result()` return NaN
  (the alignment denominator is zero) — that is a property of linear CKA, not
  an error.
- All accumulator arithmetic happens on detached CPU `float64` copies, so the
  loop never grows the autograd graph and never keeps GPU memory alive.
- For **single-stream** statistics (means, norms, covariances, PCA, quantiles,
  top-k) over a dataloader, prefer the one-call driver
  `tl.stats.aggregate(model, dataloader, metrics)`; paired-input statistics
  like `CKA`/`CrossCovariance` take the explicit loop shown here because each
  update consumes two activation batches at once.